In [1]:
%%time
import joblib
import cftime
from datetime import datetime
from datetime import timedelta
import numpy as np
import pandas as pd
import xarray as xr
from sklearn.preprocessing import minmax_scale
from matplotlib.ticker import PercentFormatter
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import optuna
from optuna.exceptions import TrialPruned
import gc
import os, sys
sys.path.insert(0,'/home/mingyu_seong/Works/func/python')
import mg

CPU times: user 1.16 s, sys: 152 ms, total: 1.31 s
Wall time: 16.8 s


/home/mingyu_seong/Works/lib/anaconda3/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read annual time series

In [2]:
#--- Input period
syr = '2001'
eyr = '2024'

#--- Clim period
clim_syr = 2001
clim_eyr = 2020

ann_feature_names_ordered = [
    'T_clim', 'P_clim', 'SM_clim', 'SW_clim',
    'T_anom', 'P_anom', 'SM_anom', 'SW_anom']

#--- Input directories
idir = '../../../_nc/input/OBS/background/r' + str(clim_syr)[2:4] + str(clim_eyr)[2:4] + '/'
idir_clim = idir + 'clim/'
idir_anom = idir + 'anom/'

#--- Input files (monthly time series)
fin_ann_tas  = 'tas_mon_ERA5_V0_KOR_01deg_2001-2024.nc'
fin_ann_prcp = 'pr_mon_GPM_L3_KOR_01deg_2001-2024.nc'
fin_ann_sm   = 'sfcsm_mon_ERA5_V0_KOR_01deg_2001-2024.nc'
fin_ann_rsds = 'rsds_mon_MODIS_C62_KOR_01deg_2001-2024.nc'

data_files = {
    'T_clim':  (idir_clim, fin_ann_tas,  'clim'),
    'P_clim':  (idir_clim, fin_ann_prcp, 'clim'),
    'SM_clim': (idir_clim, fin_ann_sm,   'clim'),
    'SW_clim': (idir_clim, fin_ann_rsds, 'clim'),
    'T_anom':  (idir_anom, fin_ann_tas,  'anom'),
    'P_anom':  (idir_anom, fin_ann_prcp, 'anom'),
    'SM_anom': (idir_anom, fin_ann_sm,   'anom'),
    'SW_anom': (idir_anom, fin_ann_rsds, 'anom'),
}

#--- Read data
raw_data_arrays = {}
for var_name, (path, filename, da_name) in data_files.items():
    full_path = os.path.join(path, filename)
    try:
        with xr.open_dataset(full_path) as ncid:
            da = ncid[da_name].copy()
            raw_data_arrays[var_name] = da.groupby('time.year').mean()  # Annual mean
    except FileNotFoundError:
        print(f"  오류: 파일이 존재하지 않습니다: {full_path}")
        sys.exit(1) # 파일이 없으면 스크립트 중단
ann_features_ds = xr.Dataset()
for feature_name in ann_feature_names_ordered:
    da = raw_data_arrays[feature_name]
    ann_features_ds[feature_name] = da

#--- Label (gC/m^2/8days)
idir = '/home/share/KCP/2025_final_input/_nc/day/MODIS/gpp/'
fin = 'gpp_day_MODIS_C61_KOR_01deg_20000101-20241231.nc'
ncid = xr.open_dataset(idir+fin)
tmp_val = ncid['gpp']
tmp_val = tmp_val.rename({'longitude': 'lon', 'latitude': 'lat'})
weights = np.cos(np.deg2rad(tmp_val['lat']))
weights.name = "weights"
tmp_val = tmp_val.sel(time=slice(syr,eyr))                            # 2015.04.01-2024.12.31
gpp  = tmp_val.convert_calendar('noleap')                             # Remove leap day
gpp.data = gpp.data * 1000                                            # gC/m^2/8days
ncid.close() ; del ncid,tmp_val
gpp = gpp.weighted(weights).mean(("lat", "lon")).groupby('time.year').mean()
ann_gpp = gpp.rename({'year': 'time'})
ann_features_ds['gpp'] = ann_gpp
del gpp

### Read spatial GPP (8-day; gC/m^2/8-day)

In [ ]:
#--- Random seed
rs = 42

#--- Analysis period
csdate = '2001-01-01'
cedate = '2024-12-31'

#--- Read data
idir = '/home/share/KCP/2025_final_input/_nc/day/MODIS/gpp/'
fin = 'gpp_8day_MODIS_C61_KOR_01deg_20000101-20241231.nc'
ncid = xr.open_dataset(idir+fin)
tmp_val = ncid['gpp']
clon = ncid['longitude'].data
clat = ncid['latitude'].data
nlon = len(clon)
nlat = len(clat)
tmp_val = tmp_val.sel(time=slice(csdate,cedate))                      # 2015.04.01-2024.12.31
val  = tmp_val.convert_calendar('noleap')                             # Remove leap day
val.data = val.data * 1000 #/8                                        # gC/m^2/8days
ncid.close() ; del ncid,tmp_val

tmp_time = []
for t in val.coords['time'].values:
    new_time = cftime.DatetimeNoLeap(t.year, t.month, t.day, 0, 0, 0)
    tmp_time.append(new_time)
val = val.assign_coords(time=tmp_time)
modis_time = val.time
gpp = val.rename({'longitude': 'lon', 'latitude': 'lat'})

In [4]:
# #--- Load Optuna results
# fin = '12fold_mean_lgbm_gpp_8days_raw_optuna.db'
# study = optuna.load_study(study_name='lgbm_gpp',
#                           storage='sqlite:///./optuna/' + fin ) 
# trial = study.best_trial
# trial

FrozenTrial(number=81, state=1, values=[4.626796234760511], datetime_start=datetime.datetime(2025, 9, 23, 6, 10, 22, 785819), datetime_complete=datetime.datetime(2025, 9, 23, 6, 16, 5, 799733), params={'n_estimators': 920, 'learning_rate': 0.059652678039804244, 'num_leaves': 143, 'max_depth': 15, 'min_child_samples': 36, 'subsample': 0.9001275164044255, 'colsample_bytree': 0.8055889507979611, 'reg_alpha': 0.3941709918223236, 'reg_lambda': 0.2217710148329864}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=1000, log=False, low=100, step=1), 'learning_rate': FloatDistribution(high=0.3, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=150, log=False, low=31, step=1), 'max_depth': IntDistribution(high=15, log=False, low=5, step=1), 'min_child_samples': IntDistribution(high=100, log=False, low=20, step=1), 'subsample': FloatDistribution(high=1.0, log=False, low=0.7, step=None), 'colsample_bytree': FloatDistribut

In [4]:
#--- Load Optuna results
fin = '12fold_mean_lgbm_gpp_8days_raw_optuna.db'
study = optuna.load_study(study_name='lgbm_gpp',
                          storage='sqlite:///./optuna/' + fin ) 
trial = study.best_trial

#--- Input variables
feature_names_ordered = [
    'T_clim', 'P_clim', 'SM_clim', 'SW_clim', 'Elev',
    'T_anom', 'P_anom', 'SM_anom', 'SW_anom', 
    'cos_lon', 'sin_lon', 'cos_lat', 'sin_lat', 'cal_day'
]

#--- Directories
idir = '/home/share/KCP/2025_final_input/_nc/input/r' + str(clim_syr)[2:4] + str(clim_eyr)[2:4] + '/OBS/'
idir_clim = idir + 'clim/'
idir_anom = idir + 'anom/'
idir_raw  = idir + 'auxiliary/'
idir_label= idir + 'anom/'

#--- Output directories
odir = '../../../_nc/prediction/OBS/GPP/variation/'
os.makedirs(odir, exist_ok=True) 

#--- Period
sdate = '2001-01-01'
edate = '2024-12-31'

#--- Input files (Climatology & Anomaly)
fin_tas  = 'tas_day_ERA5_V0_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
fin_prcp = 'pr_day_GPM_L3_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
fin_sm   = 'sfcsm_day_ERA5_V0_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
fin_rsds = 'rsds_day_MODIS_C62_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
fin_elev = 'elev_day_srtm30plus_v11_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'

#--- Input files (Auxiliary: Lon/Lat & Calendar day)
fin_loc  = 'lonlat_day_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
fin_cday = 'calday_day_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'

data_files = {
    'T_clim':  (idir_clim, fin_tas,  'clim'),
    'P_clim':  (idir_clim, fin_prcp, 'clim'),
    'SM_clim': (idir_clim, fin_sm,   'clim'),
    'SW_clim': (idir_clim, fin_rsds, 'clim'),
    'Elev':    (idir_raw, fin_elev,  'clim'), 
    'T_anom':  (idir_anom, fin_tas,  'anom'),
    'P_anom':  (idir_anom, fin_prcp, 'anom'),
    'SM_anom': (idir_anom, fin_sm,   'anom'),
    'SW_anom': (idir_anom, fin_rsds, 'anom'),
    'cal_day': (idir_raw, fin_cday,'calday'),
    'cos_lon': (idir_raw, fin_loc, 'coslon'),
    'sin_lon': (idir_raw, fin_loc, 'sinlon'),
    'cos_lat': (idir_raw, fin_loc, 'coslat'),
    'sin_lat': (idir_raw, fin_loc, 'sinlat'),
}

#--- Read data
raw_data_arrays = {}
for var_name, (path, filename, da_name) in data_files.items():
    full_path = os.path.join(path, filename)
    try:
        with xr.open_dataset(full_path) as ncid:
            da = ncid[da_name]
            if 'time' in da.dims:
                da = da.sel(time=slice(sdate, edate))
                new_times = []
                for t in da.coords['time'].values:
                    new_time = cftime.DatetimeNoLeap(t.year, t.month, t.day, 0, 0, 0)
                    new_times.append(new_time)
                da = da.assign_coords(time=new_times)

            #--- 8-day average
            da_accum = []
            for t_modis in modis_time.values:
                start_date = cftime.DatetimeNoLeap(t_modis.year, t_modis.month, t_modis.day)
                end_date = start_date + timedelta(days=7)
                data_8day = da.sel(time=slice(start_date, end_date))
                if (var_name == "Elev") or (var_name == "cal_day") or    \
                   (var_name == "sin_lon") or (var_name == "sin_lat") or \
                   (var_name == "cos_lon") or (var_name == "cos_lat"):
                    data_accum = data_8day.sel(time=start_date)
                else:
                    data_accum = data_8day.mean(dim='time',skipna=False)
                da_accum.append(data_accum)

            #--- Add MODIS time info
            da_accum = xr.concat(da_accum, dim='time')
            da_accum['time'] = modis_time
            raw_data_arrays[var_name] = da_accum
            
    except FileNotFoundError:
        print(f"  오류: 파일이 존재하지 않습니다: {full_path}")
        sys.exit(1) # 파일이 없으면 스크립트 중단
        
#--- Remove background changes
features_ds = xr.Dataset()
for feature_name in feature_names_ordered:
    da = raw_data_arrays[feature_name].copy()
    if feature_name in ann_feature_names_ordered:
        da.data = da.data - np.repeat(ann_features_ds[feature_name].data, 46)[:, np.newaxis, np.newaxis]  # variation
    features_ds[feature_name] = da
label_da = gpp.copy()
label_da.data = label_da.data - np.repeat(ann_features_ds['gpp'].data, 46)[:, np.newaxis, np.newaxis]     # variation

#--- Combined inputs
print(f"데이터셋으로 통합 및 스태킹 중...")
combined_features_da = features_ds.to_array(dim='variable')

데이터셋으로 통합 및 스태킹 중...


In [ ]:
for ifold in np.arange(12):
    #--- Initialization
    train_start  = None ; train_end  = None
    train_start0 = None ; train_end0 = None
    train_start1 = None ; train_end1 = None
    valid_start  = None ; valid_end  = None
    valid_start0 = None ; valid_end0 = None
    valid_start1 = None ; valid_end1 = None
    test_start   = None ; test_end   = None
    
    # Test: 17-yr / Validation: 5-yr / Test: 2-yr  (Same as FFNN)
    if ifold == 0:
        train_start = '2003' ; train_end = '2019'
        valid_start = '2020' ; valid_end = '2024'
        test_start  = '2001' ; test_end  = '2002'
    elif ifold == 1:
        train_start  = '2005' ; train_end  = '2021'
        valid_start0 = '2022' ; valid_end0 = '2024'
        valid_start1 = '2001' ; valid_end1 = '2002'
        test_start   = '2003' ; test_end   = '2004'
    elif ifold == 2:   
        train_start  = '2005' ; train_end  = '2023'
        valid_start0 = '2024' ; valid_end0 = '2024'
        valid_start1 = '2001' ; valid_end1 = '2004'
        test_start   = '2005' ; test_end   = '2006'
    elif ifold == 3:   
        train_start0 = '2001' ; train_end0 = '2001'
        train_start1 = '2009' ; train_end1 = '2024'
        valid_start  = '2002' ; valid_end  = '2006'
        test_start   = '2007' ; test_end   = '2008'
    elif ifold == 4:   
        train_start0 = '2001' ; train_end0 = '2003'
        train_start1 = '2011' ; train_end1 = '2024'
        valid_start  = '2004' ; valid_end  = '2008'
        test_start   = '2009' ; test_end   = '2010'
    elif ifold == 5:   
        train_start0 = '2001' ; train_end0 = '2005'
        train_start1 = '2013' ; train_end1 = '2024'
        valid_start  = '2006' ; valid_end  = '2010'
        test_start   = '2011' ; test_end   = '2012'
    elif ifold == 6:   
        train_start0 = '2001' ; train_end0 = '2007'
        train_start1 = '2015' ; train_end1 = '2024'
        valid_start  = '2008' ; valid_end  = '2012'
        test_start   = '2013' ; test_end   = '2014'
    elif ifold == 7:   
        train_start0 = '2001' ; train_end0 = '2009'
        train_start1 = '2017' ; train_end1 = '2024'
        valid_start  = '2010' ; valid_end  = '2014'
        test_start   = '2015' ; test_end   = '2016'
    elif ifold == 8:   
        train_start0 = '2001' ; train_end0 = '2011'
        train_start1 = '2019' ; train_end1 = '2024'
        valid_start  = '2012' ; valid_end  = '2016'
        test_start   = '2017' ; test_end   = '2018'
    elif ifold == 9:   
        train_start0 = '2001' ; train_end0 = '2013'
        train_start1 = '2021' ; train_end1 = '2024'
        valid_start  = '2014' ; valid_end  = '2018'
        test_start   = '2019' ; test_end   = '2020'
    elif ifold == 10:   
        train_start0 = '2001' ; train_end0 = '2015'
        train_start1 = '2023' ; train_end1 = '2024'
        valid_start  = '2016' ; valid_end  = '2020'
        test_start   = '2021' ; test_end   = '2022'
    elif ifold == 11:   
        train_start = '2001' ; train_end = '2017'
        valid_start = '2018' ; valid_end = '2022'
        test_start  = '2023' ; test_end  = '2024'

    #--- Train
    if train_start0 is not None and train_start1 is not None:
        train_features_da_part0 = combined_features_da.sel(time=slice(train_start0, train_end0))      # First period
        train_label_da_part0 = label_da.sel(time=slice(train_start0, train_end0))
        train_features_da_part1 = combined_features_da.sel(time=slice(train_start1, train_end1))      # Second period
        train_label_da_part1 = label_da.sel(time=slice(train_start1, train_end1))
        train_features_da = xr.concat([train_features_da_part0, train_features_da_part1], dim='time') # Merge periods
        train_label_da = xr.concat([train_label_da_part0, train_label_da_part1], dim='time')
    else:
        train_features_da = combined_features_da.sel(time=slice(train_start, train_end))
        train_label_da = label_da.sel(time=slice(train_start, train_end))

    #--- Validation
    if valid_start0 is not None and valid_start1 is not None:
        valid_features_da_part0 = combined_features_da.sel(time=slice(valid_start0, valid_end0))      # First period 
        valid_label_da_part0 = label_da.sel(time=slice(valid_start0, valid_end0))
        valid_features_da_part1 = combined_features_da.sel(time=slice(valid_start1, valid_end1))      # Second period 
        valid_label_da_part1 = label_da.sel(time=slice(valid_start1, valid_end1))
        valid_features_da = xr.concat([valid_features_da_part0, valid_features_da_part1], dim='time') # Merge periods
        valid_label_da = xr.concat([valid_label_da_part0, valid_label_da_part1], dim='time')
    else:
        valid_features_da = combined_features_da.sel(time=slice(valid_start, valid_end))
        valid_label_da = label_da.sel(time=slice(valid_start, valid_end))

    test_features_da = combined_features_da.sel(time=slice(test_start, test_end))      
    test_label_da = label_da.sel(time=slice(test_start, test_end))

    stacked_train_features_list, stacked_valid_features_list, stacked_test_features_list = [],[],[]
    for i, feature_name in enumerate(feature_names_ordered):
        stacked_train_feature_array = train_features_da[i,:,:,:].stack(sample=('time', 'lat', 'lon')).values
        stacked_valid_feature_array = valid_features_da[i,:,:,:].stack(sample=('time', 'lat', 'lon')).values
        stacked_test_feature_array  = test_features_da[i,:,:,:].stack(sample=('time', 'lat', 'lon')).values
        stacked_train_features_list.append(stacked_train_feature_array)
        stacked_valid_features_list.append(stacked_valid_feature_array)
        stacked_test_features_list.append(stacked_test_feature_array)
    
    #--- Separate data (train / validation / test)
    train_x = np.transpose(np.vstack(stacked_train_features_list))
    valid_x = np.transpose(np.vstack(stacked_valid_features_list))
    test_x  = np.transpose(np.vstack(stacked_test_features_list))
    train_y = train_label_da.stack(sample=('time', 'lat', 'lon')).values
    valid_y = valid_label_da.stack(sample=('time', 'lat', 'lon')).values
    test_y  = test_label_da.stack(sample=('time', 'lat', 'lon')).values
    
    #--- Remove NaN according to label
    idx = ~np.isnan(train_y)
    train_x = train_x[idx]
    train_y = train_y[idx]
    del idx

    idx = ~np.isnan(valid_y)
    valid_x = valid_x[idx]
    valid_y = valid_y[idx]
    del idx
       
    #--- Start Optuna
    n_threads = 8
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'importance_type': 'gain',
        'random_state': rs,         
        'n_estimators': trial.params.get('n_estimators'),
        'learning_rate': trial.params.get('learning_rate'),
        'num_leaves': trial.params.get('num_leaves'),
        'max_depth': trial.params.get('max_depth'),
        'min_child_samples': trial.params.get('min_child_samples'),
        'subsample': trial.params.get('subsample'),
        'colsample_bytree': trial.params.get('colsample_bytree'),
        'reg_alpha': trial.params.get('reg_alpha'),
        'reg_lambda': trial.params.get('reg_lambda'),
        'num_threads': n_threads,
        'verbosity': -1,}

    # LightGBM 모델 초기화 및 학습 (Early Stopping 포함)
    model = lgb.LGBMRegressor(**params)
    model.fit(train_x, train_y,
              eval_set=[(valid_x, valid_y)],
              eval_metric='rmse',
              callbacks=[lgb.early_stopping(stopping_rounds=150)])

    #--- Save model
    model_path = './lightGBM/lgbm_model_' + test_start.replace('-','') + \
                 '-' + test_end.replace('-','') + \
                 '_fold' + str(ifold).zfill(2) + \
                 '_seed' + str(rs) + '.joblib'
    joblib.dump(model, model_path)

    #--- Inference
    y_pred = model.predict(test_x)
    mask = ~np.isnan(test_y) & ~np.isnan(y_pred)
    pred = y_pred.reshape(len(test_label_da.time.data),len(test_label_da.lat.data),len(test_label_da.lon.data))
    pred[np.isclose(pred, pred[0,0,0] , atol=1e-6 )] = np.nan

    #--- Save
    pred_data = xr.DataArray(pred,
                            dims=['time', 'lat', 'lon'],
                            coords={'time': test_label_da.time.data,
                                    'lat': test_label_da.lat.data, 'lon': test_label_da.lon.data},
                            attrs={'long_name': 'Predicted GPP',
                                   'units': 'gC/m^2/8days'})#, '_FillValue': 1e+30})
    
    raw_data = xr.DataArray(test_label_da,
                            dims=['time', 'lat', 'lon'],
                            coords={'time': test_label_da.time.data,
                                    'lat': test_label_da.lat.data, 'lon': test_label_da.lon.data},
                            attrs={'long_name': 'Predicted GPP',
                                   'units': 'gC/m^2/8days'})#, '_FillValue': 1e+30})

    ds = xr.Dataset()
    ds['pred'] = pred_data
    ds['raw'] = raw_data
    ds['lon'].attrs['long_name'] = "longitude"
    ds['lat'].attrs['long_name'] = "latitude"
    ds['lon'].attrs['axis'] = "X"
    ds['lat'].attrs['axis'] = "Y"
    ds.to_netcdf(odir + 'predicted_GPP_8day_LGBM_' + \
                 test_start.replace('-','') + '-' + \
                 test_end.replace('-','') + '_fold' + \
                 str(ifold).zfill(2) + '_seed' + str(rs) + '.nc')
    del train_x, train_y, test_x, test_y
    del pred_data, raw_data, ds

In [6]:
#--- Merging times
fou = 'predicted_GPP_8day_LGBM_' + sdate.replace('-','') + '-' + edate.replace('-','') + '_seed' + str(rs)+'.nc'
ftmp = mg.get_files(pin=odir,fin='predicted_GPP_8day_LGBM_*_fold*seed*.nc')
flist = [os.path.join(odir, filename ) for filename  in ftmp]
mg.cdo('mergetime',input=flist, output=odir+fou, options='-s -L -O')

"'../../../_nc/prediction/OBS/GPP/variation/predicted_GPP_8day_LGBM_20010101-20241231_seed42.nc'"